Written by Diya Hamada. April 2025.

Employs csv files produced by forage_model_SP25.ipynb for initial single-layer ML confusion matrix investigation. Includes factor exploration data visualizations.

In [ ]:
# Import libraries + data
import pandas as pd
import numpy as np
import datetime as dt
from sklearn.utils import resample

# read in CSVs generated in forage_model_SP25.ipynb
data = pd.DataFrame()
for day in range(0, 30): # select range
    newData = pd.read_csv(f'day{day}.csv')
    data = pd.concat([data, newData])

# Tidying
result = data.drop(columns=['Unnamed: 0'])

# DF ready for analysis!
result = result.sort_values(by=['uid', 'datetime'])
result = result[['uid', 'Age', 'Day (Seasonal Cycle)', 'YYYYMMDD', 'Second (Daily Cycle)', 'HHMMSS', 'mins_since_df_visit', 'status']]
result = result.drop_duplicates()
result

In [ ]:
result.status.value_counts()

In [ ]:
# Balancing data
df = pd.concat([
    resample(result[result['status'] == 'no change'], n_samples=result.status.value_counts()[1], random_state=42),
    result[result['status'] == 'leave']
]).sample(frac=1, random_state=42).reset_index(drop=True)
df.status.value_counts()

In [ ]:
# quick switch from no change and leave to 0 and 1
df['status'] = df['status'].map({'no change': 0, 'leave': 1})

# Split Train and Test Data
from sklearn.model_selection import train_test_split

y = df['status']
X = df[['Age','Day (Seasonal Cycle)', 'Second (Daily Cycle)', 'mins_since_df_visit']]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.33, random_state=25)

In [ ]:
df.to_csv('df.csv')

In [ ]:
# more libraries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
import seaborn as sns
from sklearn.metrics import confusion_matrix,precision_score,recall_score,f1_score,accuracy_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# snatched straight from Padma <3
def model_performance(model,x_train,x_test, y_train,y_test):
    print(f"{model} Performance:\n")
    
    y_pred = model.predict(x_test)
    cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
    sns.set(rc = {'figure.figsize':(6,4)})
    sns.heatmap(cm, cmap = 'Blues', annot = True, fmt = 'd', linewidths = 2, linecolor='black', clip_on = False, cbar = False, annot_kws = {'fontsize': 15}, 
    yticklabels = ['No Forage', 'Forage'], xticklabels = ['Predicted no forage', 'Predicted forage'])
    plt.show()
   
    
    precision = precision_score(y_test,y_pred)
    recall = recall_score(y_test,y_pred)
    accuracy = accuracy_score(y_test,y_pred)
    f1 = f1_score(y_test,y_pred)

    cross_val_score_insample = cross_val_score(model,x_train,y_train,cv=5,scoring='accuracy').mean()
    cross_val_score_outsample = cross_val_score(model,x_test,y_test,cv=5,scoring='accuracy').mean()
    
    print('Precision Score:',precision)
    print("Recall Score:",recall)
    print("Accuracy Score:",accuracy)
    print('F1 Score:',f1)
    print("Cross Val Score Insample",cross_val_score_insample)
    print("Cross Val Score Outsample",cross_val_score_outsample)
    
    return model,precision,recall,accuracy,cross_val_score_insample,cross_val_score_outsample

## Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
y_pred = gnb.fit(X_train, y_train).predict(X_test)

performance_nb = model_performance(gnb, X_train, X_test, y_train, y_test)

## KNN

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

knn_temp_model = KNeighborsClassifier()
param_grid={'weights' : ['uniform','distance'],'n_neighbors' : (np.arange(9)+2)}
knn_grid = GridSearchCV(knn_temp_model,param_grid,cv=5,scoring='accuracy',verbose=1)
grid_results = knn_grid.fit(X_train,y_train)
best_k = knn_grid.best_params_['n_neighbors']
print("\t Ideal K-Val: {}".format(best_k))
print("\t Accuracy of Tuned Training Data: {:.2f}%".format(grid_results.best_score_ * 100))

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train, y_train)
performance_knn = model_performance(knn_model,X_train, X_test, y_train, y_test)

## Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=100)
y_pred = lr.fit(X_train, y_train).predict(X_test)
print("Number of mislabeled points out of a total %d points : %d" % (X_test.shape[0], (y_test != y_pred).sum()))
performance_lr = model_performance(lr,X_train, X_test, y_train, y_test)

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier()
y_pred = rfc.fit(X_train, y_train).predict(X_test)
print("Number of mislabeled points out of a total %d points : %d" % (X_test.shape[0], (y_test != y_pred).sum()))
performance_rfc = model_performance(rfc,X_train, X_test, y_train, y_test)

In [ ]:
feature_names = X.columns
importances = rfc.feature_importances_
feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Gini Importance': importances}).sort_values('Gini Importance', ascending=False) 
print(feature_imp_df)

# Create a bar plot for feature importance
plt.figure(figsize=(8, 4))
plt.barh(feature_names, importances, color='darkblue', edgecolor='black')
plt.xlabel('Gini Importance')
plt.title('Feature Importance - Gini Importance')
plt.gca().invert_yaxis()  # Invert y-axis for better visualization
plt.show()

In [ ]:
'''from sklearn.ensemble import RandomForestClassifier

yUnbalanced = df_unbalanced['status']
XUnbalanced = df_unbalanced[['Age','Day (Seasonal Cycle)', 'Second (Daily Cycle)', 'mins_since_df_visit']]

rfc = RandomForestClassifier()
y_pred = rfc.fit(X_train, y_train).predict(XUnbalanced)
print("Number of mislabeled points out of a total %d points : %d" % (XUnbalanced.shape[0], (yUnbalanced != y_pred).sum()))
performance_rfc = model_performance(rfc, X_train, XUnbalanced, y_train, yUnbalanced)
'''

## SVM

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel = 'linear')
y_pred = svm.fit(X_train, y_train).predict(X_test)
print("Number of mislabeled points out of a total %d points : %d" % (X_test.shape[0], (y_test != y_pred).sum()))
performance_svm = model_performance(svm,X_train, X_test, y_train, y_test)

- include rest of days
- analysis for factors individually + weights
- neural network (include variation in parameters/process)
- try grid method for random forest+similar, not just knn

# Factor Exploration

In [ ]:
leaveEvents = result[result["status"] == "leave"].sort_values(by='Age')
leaveEvents

In [ ]:
import matplotlib.pyplot as plt

leaveEvents['Age'].value_counts(sort=False).plot(kind='bar')

plt.xlabel('Age (days)')
plt.ylabel('Count')
plt.title('Ages of Leaving Bees')

plt.show()

In [ ]:
secs = round(leaveEvents['Second (Daily Cycle)']/100000, 1)
secs.value_counts(sort=False).plot(kind='bar')

plt.xlabel('second of day/100000')
plt.ylabel('Count')

plt.show()

In [ ]:
plt.boxplot(leaveEvents['Second (Daily Cycle)'], labels = ["Second of the Day"])
plt.show()

In [ ]:
plt.boxplot(leaveEvents['Age'], labels = ["Age"])
plt.show()

In [ ]:
mins = round(leaveEvents['mins_since_df_visit']/1000, 1)
mins.value_counts(sort=False).plot(kind='bar')

plt.xlabel('mins since df visit/1000')
plt.ylabel('Count')

plt.show()

In [ ]:
plt.boxplot(leaveEvents['mins_since_df_visit'], labels = ["lastdfvisit"])
plt.show()